# Guide04: Beyond Straight Lines

Fourth of six notebooks on one running example: predicting Ames house prices. This one is the first half of **Stage 9** of the [13-stage workflow](../Guide00_Supervised-ML_Linear_Regression_end-to-end_workflow.md): can a more flexible model do better than `Guide03`'s baseline, and what does that flexibility cost? `Guide05` finishes Stage 9 with the fix.

**Prerequisite:** `Guide03_Supervised-ML_Linear-Regression_Baseline-Model-and-Evaluation.ipynb`.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
plt.rcParams["figure.figsize"] = (7, 4)


---
## Recap: Where `Guide03` Left Off

Rebuild the training data and the 5-fold cross-validation setup, exactly as `Guide03` used them.


In [ ]:
from pipeline.ames_workflow import load_ames, clean_ames, split_ames, add_engineered_features, build_preprocessor

X_train, X_test, y_train, y_test = split_ames(clean_ames(load_ames()))
X_train = add_engineered_features(X_train)
print(f"train: {X_train.shape}   test (locked, not used below): {X_test.shape}")


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline

kf = KFold(n_splits=5, shuffle=True, random_state=42)


def evaluate(name, model, X=X_train, y=y_train):
    preds = cross_val_predict(model, X, y, cv=kf)
    mae = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)
    print(f"{name:42s} MAE=${mae:>10,.0f}   RMSE=${rmse:>10,.0f}   R2={r2:8.3f}")
    return preds


def make_model(poly_cols=None, degree=2):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("prep", build_preprocessor(X_train, poly_cols=poly_cols, degree=degree)),
            ("model", LinearRegression()),
        ]),
        func=np.log, inverse_func=np.exp,
    )


baseline_preds = evaluate("Guide03 baseline (no polynomial terms)", make_model())


`Guide03`'s score to beat: about \$14,100 MAE, $R^2 \approx 0.93$. **This notebook uses plain `LinearRegression` throughout, deliberately not Ridge or Lasso** -- those are `Guide05`'s Stage 9.2. The point here is to see what happens *without* a penalty on complexity, so that `Guide05`'s fix means something.


---
## Is the Relationship Really Linear?

`Guide01` found `OverallQual` and `GrLivArea` as the two strongest predictors of price. A straight line is only the right tool if their relationship with price is roughly straight. Look at the mean price within bins of each:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

qual_means = y_train.groupby(X_train["OverallQual"]).mean()
axes[0].plot(qual_means.index, qual_means.values, marker="o")
axes[0].set(title="Mean price by OverallQual", xlabel="OverallQual", ylabel="mean SalePrice")

area_bins = pd.cut(X_train["GrLivArea"], bins=8)
area_means = y_train.groupby(area_bins, observed=True).mean()
axes[1].plot(range(len(area_means)), area_means.values, marker="o")
axes[1].set_xticks(range(len(area_means)))
axes[1].set_xticklabels([f"{b.left:.0f}" for b in area_means.index], rotation=45)
axes[1].set(title="Mean price by GrLivArea bin", xlabel="GrLivArea (bin start)", ylabel="mean SalePrice")

plt.tight_layout()
plt.show()


Both curve upward rather than tracing a straight line -- the jump from `OverallQual` 9 to 10 is much bigger than the jump from 2 to 3, and the same acceleration shows up in the largest `GrLivArea` bins. That is a real signal for polynomial terms, not an excuse to add them blindly. Whether *acting* on that signal actually helps is a separate question -- next.


---
## A Modest Try: Degree 2 on Four Key Numerics

`PolynomialFeatures(degree=2)` on a handful of columns adds each one's square *and* every pairwise product between them -- interaction terms, not just curvature, all in one step:


In [ ]:
key_cols = ["OverallQual", "GrLivArea", "GarageArea", "TotalBsmtSF"]

poly_preds = evaluate("degree 2 on 4 key numerics", make_model(poly_cols=key_cols, degree=2))


Essentially no improvement over the plain baseline -- if anything, slightly worse. The curvature we saw above is real, but a linear model with a rich enough set of *other* features (neighborhood, quality grades, square footage, ...) was apparently already capturing most of it indirectly. Curvature visible in one feature at a time does not automatically mean a polynomial term earns its keep once the whole model is considered.


---
## Bias and Variance: Train Error vs. Cross-Validated Error

Push the degree further, on the same four columns, and compare the error the model reports on the data it was fit on against its cross-validated error on data it was not fit on:


In [ ]:
results = []
for degree in (1, 2, 3):
    model = make_model(poly_cols=key_cols, degree=degree)
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    cv_mae = mean_absolute_error(y_train, cross_val_predict(model, X_train, y_train, cv=kf))
    results.append({"degree": degree, "train_MAE": train_mae, "cv_MAE": cv_mae})

pd.DataFrame(results).set_index("degree").round(0)


Training error keeps improving as the degree rises -- the model can always fit its own training data a little better by adding flexibility. Cross-validated error does the opposite past degree 1: it gets *worse*. That gap opening up between the two lines is what overfitting looks like in numbers, not just in a textbook diagram: extra flexibility here is being spent memorizing training-set quirks, not learning anything that generalizes.


---
## A First Hyperparameter Search

The polynomial degree is a hyperparameter -- something we choose, not something the model learns. `Guide00`'s Stage 9.3 describes searching a grid of candidate values with cross-validation instead of guessing. With one hyperparameter and three candidate values, that search is small enough to run directly:


In [ ]:
from sklearn.model_selection import GridSearchCV

grid_model = make_model(poly_cols=key_cols, degree=2)  # degree=2 here is just a placeholder; the grid overrides it
grid = GridSearchCV(
    grid_model,
    param_grid={"regressor__prep__num_poly__poly__degree": [1, 2, 3]},
    cv=kf,
    scoring="neg_mean_absolute_error",
)
grid.fit(X_train, y_train)

print("best degree found:", grid.best_params_["regressor__prep__num_poly__poly__degree"])
print(f"its cross-validated MAE: ${-grid.best_score_:,.0f}")


The search lands on degree 1 -- no polynomial terms at all -- which matches what the manual comparison above already showed. That agreement is the point: a proper search, using only training data, would have protected us from degree 3 even if we had never plotted train-vs-CV error ourselves. `Guide05` scales this same idea up to a real grid, with regularization added.


---
## The Cost of Flexibility: What Happens With Many More Features

Four polynomial columns barely moved the score. What if every numeric column gets the same treatment?


In [ ]:
all_numeric_cols = list(X_train.select_dtypes("number").columns)

for label, cols in [
    ("4 key numerics", key_cols),
    ("15 numeric columns", all_numeric_cols[:15]),
    (f"all {len(all_numeric_cols)} numeric columns", all_numeric_cols),
]:
    n_out = build_preprocessor(X_train, poly_cols=cols, degree=2).fit_transform(X_train).shape[1]
    preds = cross_val_predict(make_model(poly_cols=cols, degree=2), X_train, y_train, cv=kf)
    mae, r2 = mean_absolute_error(y_train, preds), r2_score(y_train, preds)
    print(f"{label:24s} -> {n_out:4d} output columns   MAE=${mae:.3e}   R2={r2:.3e}")

print(f"\n(for comparison: only {len(X_train)} training rows, about {int(len(X_train) * 0.8)} in each cross-validation fold)")


Read the last row's numbers as what they are, not as a typo: an MAE and $R^2$ with dozens of digits are not a *worse* model, they are a **degenerate** one. Once the number of columns (921, from squaring and cross-multiplying 37 numeric features) gets close to the number of rows in a training fold, the matrix `LinearRegression` has to invert stops being reliably invertible at all. The "coefficients" it returns are numerical noise, not a fit -- which is exactly why the predictions built from them are numerical noise too.

This is not a corner case to avoid by being careful; it is what *unregularized* linear regression does whenever flexibility is allowed to grow unchecked. `Guide05`'s regularization (Ridge, Lasso) is the fix: a penalty that keeps coefficients from exploding even when the raw problem is this ill-posed.


---
## What We Hand to `Guide05`

* **No new engineered features to keep.** Polynomial terms on the strongest predictors did not beat the `Guide03` baseline; the curvature we found is worth remembering, but the fix is not "add more polynomial terms" without control.
* **The central lesson:** flexibility (more polynomial terms, more columns) is not free, and past a point it is actively destructive without a check on complexity. `Guide05` supplies that check.
* **A working grid-search pattern** (`GridSearchCV` over a `regressor__prep__...` parameter path) that `Guide05` extends to a real grid.
* The locked test set, still untouched.


---
## Your Turn

**`encoded_car_data.csv`.** It has 35 already-encoded numeric columns and 205 rows. Before running anything: if `PolynomialFeatures(degree=2)` were applied to all 35 columns, how many output columns would there be, roughly? (Hint: think about how many pairwise products 35 columns produce, not just the 35 squared terms.) Write down a guess, then compute it and compare against the 205 available rows. Do you expect the same kind of collapse this notebook found for Ames? Try it and see.


In [ ]:
# Your turn: load encoded_car_data.csv, guess the output column count, then check it.
